# What this corpus is about

Everything below reads the exported tables. The three LLM passes run on the
command line first:

```bash
topicsmith extract      # free-form labels, one call per paper
topicsmith taxonomy     # induce the canonical areas from the pooled labels
#                       # -> now read and edit data/interim/taxonomy.yaml
topicsmith assign       # re-classify every paper against the edited taxonomy
topicsmith export
```

The editing step in the middle is the point of the tool. `assign` reads whatever
is in `taxonomy.yaml`, and that file is part of no cache key, so disagreeing
with the model costs one re-run of the third pass and no re-reading of papers.

Every figure here comes from `topicsmith.figures`, which is also what
`topicsmith figures` renders headlessly — so what you see here and what a
Makefile produces cannot drift apart.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd

from topicsmith import analysis, assemble, figures, viz
from topicsmith.config import load_config
from topicsmith.passes import taxonomy as taxonomy_pass

cfg = load_config()

# One argument re-colours every figure below. Eight validated palettes:
# ink_ember · mediterranean · ultraviolet · forest · crimson · marigold ·
# slate · classic_blue. `scale` multiplies every font size at once — raise it
# for a projector, lower it for a printed page.
viz.apply_style(theme=cfg.viz.get("theme"), scale=cfg.viz.get("scale"))

tables = assemble.load(cfg)
papers, subtopics, areas = tables["papers"], tables["subtopics"], tables["areas"]
print(f"{len(papers)} papers · {len(areas)} areas · "
      f"{subtopics['subtopic'].nunique()} distinct subtopics")

## The taxonomy

Areas with no papers are shown too: an empty area means either the taxonomy has
a category the corpus does not support, or the assignment pass is avoiding it.
Either way it is worth knowing.

In [ ]:
for row in areas.itertuples():
    seeded = "  (seeded)" if row.seeded else ""
    print(f"{row.area}  —  {row.n_papers} papers ({row.share_pct}%){seeded}")
    print(f"    {row.definition}")
    print(f"    subtopics: {row.subtopics}\n")

In [ ]:
fig = figures.areas(tables, cfg)
plt.show()

In [ ]:
# Papers often straddle two areas. The secondary assignment shows which pairs.
pairs = analysis.area_pairs(papers)
display(pairs if len(pairs) else "Every paper sits squarely in one area.")

## Subtopics

The finer labels, as assigned in the third pass. If a subtopic here does not
appear under its area in `taxonomy.yaml`, the model added it because nothing
listed fitted — worth reading as a signal that an area's subtopic list is thin.

In [ ]:
fig = figures.subtopics(tables, cfg, top=20)
plt.show()

In [ ]:
fig = figures.subtopic_cloud(tables, cfg)
plt.show()

## How the subtopics cluster

Raise `min_papers` until the map is readable — if the labels end up in a lattice
of crossing leader lines, the fix is plotting less, not a better label solver.

In [ ]:
fig = figures.subtopic_map(tables, cfg, min_papers=3)
if fig is None:
    print("No subtopic co-occurs on enough papers at this threshold. Try min_papers=2.")
plt.show()

## Against your own metadata

One heatmap per column listed in `metadata.facets`. Nothing here renders if you
have no `metadata.csv` — the topic analysis does not depend on it.

In [ ]:
for column in cfg.metadata.get("facets") or []:
    fig = figures.facet(tables, cfg, column)
    if fig is None:
        print(f"no usable values in {column!r}")
        continue
    plt.show()

In [ ]:
fig = figures.contributions(tables, cfg)
plt.show()

# One chart per field declared in topics.extra_fields.
from topicsmith.schemas import extra_fields

for field in extra_fields(cfg):
    fig = figures.extra_field(tables, cfg, field["name"])
    if fig is not None:
        plt.show()

## How much should you believe this?

Two checks. The stability report re-induces the taxonomy several times at a
non-zero temperature: high overlap means the areas are a property of the corpus
rather than of one sampling run. The review table is the assignments the model
was least sure of — the fastest fifteen minutes you can spend on this analysis.

In [ ]:
path = cfg.paths.stability
display(pd.read_csv(path)) if path.is_file() else print("Run `topicsmith stability`.")

In [ ]:
review = cfg.paths.review
if review.is_file():
    flagged = pd.read_csv(review)
    display(flagged[["paper_id", "title", "main_area", "confidence", "why"]].head(20))
else:
    print("No low-confidence assignments.")

## Everything at once

The same call `topicsmith figures` makes. Writes PNG and SVG, with a transparent
background so they go full-bleed onto a slide of any colour.

In [ ]:
for path in figures.render_all(tables, cfg, close=False):
    print(path)